# Create or append Virtual Icechunk Store from NOAA CDR Snow Cover Extent

* Source data: `s3://noaa-cdr-snow-cover-ext-north-pds/` (public, anonymous access)
* Destination: Icechunk virtual store written to the **moeindst** bucket
* Follows the same file-set diff approach as `taranto-icechunk-append.ipynb`:
  1. **`set_repo`** — filenames already committed to the Icechunk store
  2. **`set_cloud`** — filenames currently in the source bucket
  3. **`new_files`** = `set_cloud - set_repo` — only these are virtualised and appended

In [1]:
import warnings
import os
import fsspec
import pandas as pd
import xarray as xr
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import icechunk
from virtualizarr import open_virtual_dataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
from obstore.store import S3Store

In [4]:
# Destination credentials (moeindst bucket)
_ = load_dotenv(f'{os.environ["HOME"]}/dotenv/protocoast.env',override=True)
storage_endpoint = os.environ['ENDPOINT_URL']

# Destination: Icechunk store
dest_bucket     = 'moeindst'
dest_bucket_url = f's3://{dest_bucket}'
storage_name    = 'noaa-cdr-snow-icechunk-v1'

# Source: public NOAA CDR bucket (anonymous, AWS us-east-1)
source_bucket     = 'noaa-cdr-snow-cover-ext-north-pds'
source_bucket_url = f's3://{source_bucket}'
source_prefix     = ''  # files sit at the bucket root and under data/

# Filesystem for scanning source bucket (anonymous)
fs = fsspec.filesystem('s3', anon=True)

In [5]:
# Icechunk store lives in moeindst
storage = icechunk.s3_storage(
    bucket=dest_bucket,
    prefix=f'icechunk/{storage_name}',
    from_env=True,
    endpoint_url=storage_endpoint,
    region='not-used',
    force_path_style=True,
)

config = icechunk.RepositoryConfig.default()
# Virtual chunks point to the public NOAA bucket — anonymous access
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=f'{source_bucket_url}/',
        store=icechunk.s3_store(
            region='us-east-1',
            anonymous=True,
            s3_compatible=False,
        ),
    ),
)

credentials = icechunk.containers_credentials(
    {f'{source_bucket_url}/': icechunk.s3_credentials(anonymous=True)}
)

# ObjectStore registry for VirtualiZarr — anonymous reads from NOAA bucket
source_store_obj = S3Store(
    bucket=source_bucket,
    region='us-east-1',
    skip_signature=True,
)

registry = ObjectStoreRegistry({source_bucket_url: source_store_obj})
parser = HDFParser()

### Step 1: Build file sets (`set_repo` vs `set_cloud`)

In [6]:
# --- 1. Filenames already in the Icechunk repo (set_repo) ---
# We store the source filename list as a zarr attribute on commit so we can
# reconstruct set_repo without scanning virtual refs directly.
try:
    repo = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
    session = repo.readonly_session('main')
    ds_repo = xr.open_zarr(session.store, consolidated=False, chunks={})
    # Source filenames are recorded in a global attribute at write time
    set_repo = set(ds_repo.attrs.get('source_files', '').split(','))
    set_repo.discard('')
except Exception as e:
    print(f'Repo not found or empty ({e}). Will create from scratch.')
    repo = None
    set_repo = set()

print(f'set_repo: {len(set_repo)} file(s) already committed')
for f in sorted(set_repo):
    print(f'  {f}')

# --- 2. NetCDF files available in the public bucket (set_cloud) ---
# Collect .nc files from both root and data/ prefix
cloud_files = [
    p for p in fs.glob(f'{source_bucket}/**/*.nc')
    if p.endswith('.nc')
]
# Use just the filename (basename) as the key, store full s3:// URL in a map
file_map = {os.path.basename(p): f's3://{p}' for p in cloud_files}
set_cloud = set(file_map.keys())

print(f'\nset_cloud: {len(set_cloud)} file(s) found')
for f in sorted(set_cloud):
    print(f'  {f}')

Repo not found or empty (  x the repository doesn't exist
  | 
  | context:
  |    0: icechunk::repository::open
  |              at icechunk/src/repository.rs:343
  | 
  `-> the repository doesn't exist
). Will create from scratch.
set_repo: 0 file(s) already committed

set_cloud: 2 file(s) found
  nhsce_v01r01_19661004_20210405.nc
  nhsce_v01r01_19661004_20260406.nc


In [ ]:
# --- 3. Determine new files ---
new_filenames = sorted(set_cloud - set_repo)

print(f'Files to process: {len(new_filenames)}')
for f in new_filenames:
    print(f'  {f}')

### Step 2: Virtualise new files

In [ ]:
# Variables that must be loaded eagerly (coordinates used for indexing / concat)
LOADABLE = ['time', 'latitude', 'longitude', 'snow_cover_threshold']

ds_final = None

# new_filenames = new_filenames[:1]  # uncomment to test with a single file

if new_filenames:
    vds_list = []
    for fname in new_filenames:
        url = file_map[fname]
        print(f'Virtualising {fname} ...')
        vds = open_virtual_dataset(
            url,
            parser=parser,
            registry=registry,
            loadable_variables=LOADABLE,
        )
        vds_list.append(vds)

    if len(vds_list) == 1:
        ds_final = vds_list[0]
    else:
        ds_final = xr.concat(
            vds_list, dim='time', coords='minimal', compat='override', combine_attrs='override'
        )
        # Drop duplicate time steps that appear in both files (overlap between versions)
        _, unique_idx = pd.Index(ds_final.time.values).duplicated(keep='last'), None
        _, idx = pd.Index(ds_final.time.values).__class__.factorize(
            pd.Index(ds_final.time.values)
        )
        # Keep last occurrence of each timestamp so the newer file wins
        time_series = pd.Series(range(len(ds_final.time)), index=pd.DatetimeIndex(ds_final.time.values))
        keep_idx = time_series.groupby(level=0).last().values
        ds_final = ds_final.isel(time=keep_idx)

    # Record which source files this store was built from
    ds_final.attrs['source_files'] = ','.join(sorted(set_repo | set(new_filenames)))

    print(f'\nReady: {len(ds_final.time)} time steps across {len(new_filenames)} file(s)')
    print(f'  time range: {ds_final.time.values[0]} → {ds_final.time.values[-1]}')
else:
    print('No new files. Store is already up to date.')

### Step 3: Write / append to Icechunk

In [ ]:
if ds_final is not None:
    if repo is None:
        # First run — create the repo
        repo = icechunk.Repository.create(storage, config, authorize_virtual_chunk_access=credentials)
        session = repo.writable_session('main')

        print(f'Writing {len(ds_final.time)} time steps to {dest_bucket}/icechunk/{storage_name} ...')
        ds_final.virtualize.to_icechunk(session.store)

        msg = f'Initialized: {new_filenames[0]} → {new_filenames[-1]}'
    else:
        # Subsequent run — append along the time dimension
        session = repo.writable_session('main')

        print(f'Appending {len(ds_final.time)} time steps to {dest_bucket}/icechunk/{storage_name} ...')
        ds_final.virtualize.to_icechunk(session.store, append_dim='time')

        msg = f'Appended: {new_filenames[0]} → {new_filenames[-1]}'

    snapshot_id = session.commit(msg)
    print(f'Committed: "{msg}"  (snapshot {snapshot_id})')

    # Verify
    history = repo.ancestry(branch='main')
    latest = next(history)
    print(f'Latest commit [{latest.written_at}]: {latest.message}')

else:
    print('Nothing to write.')

### Step 4: Verify — read back from Icechunk

In [ ]:
repo_verify = icechunk.Repository.open(storage, config, authorize_virtual_chunk_access=credentials)
session_verify = repo_verify.readonly_session('main')
ds_verify = xr.open_zarr(session_verify.store, consolidated=False, chunks={})
print(ds_verify)
print(f'\nSource files in store: {ds_verify.attrs.get("source_files", "(not set)")}')